# 08 --- Integration: Full End-to-End Research Query

This notebook brings together all CCA patterns:
- Hub-and-spoke coordinator
- Context isolation via explicit passing
- 4-5 scoped tools per agent
- Structured error handling
- Parallel + sequential task waves
- Deterministic conflict resolution

We'll walk through a complete research query using the `economic_impact` scenario.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask, ResearchReport
from research_agents.agent.coordinator import (
    sort_tasks_into_waves, run_coordinator, build_research_report
)
from research_agents.agent.context_builder import build_subagent_context
from research_agents.agent.subagents import SUBAGENT_CONFIGS
from research_agents.data.scenarios import SCENARIOS, ResearchScenario
from research_agents.data.sources import SOURCE_RELIABILITY_RATINGS
from research_agents.models.research import SourceReliability
from tests.conftest import make_services

## The ResearchScenario Model

The project defines pre-built scenarios (in `data/scenarios.py`) that combine all the patterns we've studied:

```python
@dataclass(frozen=True)
class ResearchScenario:
    name: str
    query: str
    expected_agents: list[str]  # Which agent types should be involved
    expected_conflicts: int     # How many source contradictions
    expected_gaps: int          # How many sources will fail
    description: str
```

Three scenarios, each targeting different patterns:

| Scenario | Tests | Conflicts | Gaps |
|----------|-------|-----------|------|
| `climate_renewable` | Conflict resolution | 1 (blog vs .gov) | 0 |
| `ai_healthcare` | Error handling | 0 | 1 (404) |
| `economic_impact` | Both | 1 (+13% vs -20%) | 1 (timeout) |

## Scenario: Remote Work Economic Impact

This scenario tests both conflict resolution AND error handling.

In [ ]:
scenario = SCENARIOS['economic_impact']
print(f'Query: {scenario.query}')
print(f'Expected agents: {scenario.expected_agents}')
print(f'Expected conflicts: {scenario.expected_conflicts}')
print(f'Expected gaps: {scenario.expected_gaps}')
print(f'Description: {scenario.description}')

## Step 1: Task Decomposition (PLAN + SORT)

The coordinator decomposes the query into SubTasks. In production, the LLM does this. Here we define them explicitly to show the structure:

In [ ]:
# Decompose into subtasks (normally the LLM does this)
tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='Search for studies on remote work and productivity',
        context='Focus on 2024 data. Look for both pro and con evidence.'),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='Query remote work statistics from the database',
        context='Use the remote_work_stats table'),
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='Analyze the Stanford remote work study',
        context='Document ID: doc-remote-work-stanford'),
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='Verify productivity claims from web and document sources',
        context='Check claims about remote work productivity impact',
        depends_on=['web', 'docs']),
]

waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    print(f'Wave {i}: {[t.task_id for t in wave]} ({"parallel" if len(wave) > 1 else "sequential"})' )

## Step 2: Context Isolation Check (DELEGATE)

Each subagent gets ONLY its explicit context. Let's verify:

In [ ]:
for task in tasks:
    ctx = build_subagent_context(task)
    print(f'{task.task_id} ({task.agent_type}):')
    print(f'  Context length: {len(ctx)} chars')
    print(f'  First 100 chars: {ctx[:100]}...')
    print()

## Step 3: Run Coordinator (Mock Mode)

Using a mock client for reproducible results. The `run_coordinator()` function executes Steps 2-3 of the 6-step flow:

```python
def run_coordinator(client, services, tasks, model) -> tuple[dict, list]:
    waves = sort_tasks_into_waves(tasks)
    for wave in waves:
        for task in wave:
            config = SUBAGENT_CONFIGS[task.agent_type]
            context = build_subagent_context(task, results)
            result = run_agent_loop(client, services, context, ...)
            results[task.task_id] = result
    return results, waves
```

In [ ]:
from types import SimpleNamespace

call_count = 0
def mock_create(**kwargs):
    global call_count
    call_count += 1
    agent_responses = {
        1: 'Found 4 sources. McKinsey reports +13% productivity. Blog claims -20%.',
        2: 'Remote work stats: 9.4% fully remote, 18.2% hybrid in 2024.',
        3: 'Stanford study: hybrid workers 13% more productive, 24% higher satisfaction.',
        4: 'Verified: +13% claim supported by multiple sources. -20% claim unsupported.',
    }
    text = agent_responses.get(call_count, 'Analysis complete.')
    return SimpleNamespace(
        content=[SimpleNamespace(type='text', text=text)],
        stop_reason='end_turn',
        usage=SimpleNamespace(input_tokens=500, output_tokens=200),
    )

mock_client = SimpleNamespace(messages=SimpleNamespace(create=mock_create))
services = make_services()

results, waves = run_coordinator(mock_client, services, tasks)
for task_id, result in results.items():
    print(f'{task_id}: {result.content[:80]}...' if len(result.content) > 80 else f'{task_id}: {result.content}')

## Step 4: Conflict Resolution + Report (RESOLVE + SYNTHESIZE)

The `build_research_report()` function compiles findings, resolves conflicts, and produces a `ResearchReport`:

```python
class ResearchReport(BaseModel):
    query: str
    findings: list[SourceResult]       # What we found
    conflicts: list[ConflictRecord]     # What contradicted
    synthesis: str                      # Combined narrative
    confidence_score: float             # 0.0-1.0, adjusted by gaps/conflicts
    gaps: list[str]                     # What we couldn't reach
```

The `gaps` field is critical -- a report that says 'we could not reach Source X' is more trustworthy than one that silently omits it.

In [ ]:
# Build report with conflict resolution
conflicts = [{
    'claim': 'Remote workers are more productive',
    'sources_for': ['https://mckinsey.com/future-of-work', 'https://bls.gov/remote-work-stats'],
    'sources_against': ['https://workfromhome-blog.example.com/productivity'],
}]

reliability = {url: rel for url, rel in SOURCE_RELIABILITY_RATINGS.items()}

report = build_research_report(
    query=scenario.query,
    results=results,
    reliability_lookup=reliability,
    conflicts=conflicts,
    gaps=['https://timeout.example.com/remote-data (timeout)'],
)

print(f'Query: {report.query}')
print(f'Findings: {len(report.findings)}')
print(f'Conflicts resolved: {len(report.conflicts)}')
for c in report.conflicts:
    print(f'  - {c.claim}: {c.resolution} (confidence: {c.confidence:.2f})')
print(f'Gaps: {report.gaps}')
print(f'Confidence score: {report.confidence_score:.2f}')

## How the Pieces Connect

Here's the complete data flow we just executed:

```
ResearchQuery("remote work productivity")
  |                                              CCA Domain
  v                                              ---------
  1. PLAN: Decompose into 4 SubTasks             Agentic Architecture
  |    (web, data, docs, facts)
  v
  2. SORT: Topological sort into 2 waves          Agentic Architecture
  |    Wave 0: [web, data, docs] (parallel)
  |    Wave 1: [facts] (depends on web, docs)
  v
  3. DELEGATE: For each task:                     Context Management
  |    build_subagent_context() -> explicit string
  |    run_agent_loop() with scoped tools          Tool Design
  v
  4. EVALUATE: fact_checker verifies claims        Reliability
  v
  5. RESOLVE: conflict_resolver.py                 Reliability
  |    reliability ranking -> majority -> human
  v
  6. SYNTHESIZE: ResearchReport                    All domains
       findings + conflicts + gaps + confidence
```

## The Three Exam Walkthrough Questions

The published article walks through three representative CCA exam questions drawn from the multi-agent research scenario. You have now seen every pattern they test. Use these as cold-read self-checks.

Each question below lists the scenario, the correct answer (with a pointer to the notebook that demonstrates it), and the distractors you'll see on the actual exam along with *why* each distractor is wrong.

### Question 1 -- Context Isolation

**Scenario.** A coordinator instructs the user to "use APA citation format." The web-research subagent returns sources formatted in MLA. Why?

**Correct answer.** The APA instruction lived in the coordinator's message history but was never placed into the subagent's `task.context`. The subagent structurally never saw it. The fix is to include formatting requirements in every `SubTask.context` where they apply.

**Where to see it.** Notebook 02 (`02_context_isolation.ipynb`) -- the `run_leaky_subagent` vs `build_subagent_context(task)` cells compare the two approaches side by side.

**Distractors and why they fail:**

- *"The subagent needs a better system prompt."* The system prompt is a general persona, not a place for query-specific requirements. Putting APA in every system prompt ever written is not context engineering.
- *"Use a larger model for the subagent."* A larger model that still never sees the instruction will make the same error more fluently.
- *"Let subagents inherit coordinator context automatically."* This is the `shared_context.py` anti-pattern -- it creates token waste, attention dilution, and privacy-style leakage of other agents' results.

### Question 2 -- Tool Overload

**Scenario.** An agent configured with 18 tools repeatedly selects the wrong tool for the task. What do you change?

**Correct answer.** Decompose the single agent into specialized subagents with 4-5 focused tools each. This is an **architectural** fix -- it is not a better-descriptions problem.

**Where to see it.** Notebook 03 (`03_tool_scoping.ipynb`) -- compares `SUPER_AGENT_TOOLS` (20 on one agent) against `ALL_TOOL_SETS` (5 agents x 4 each). `tests/test_anti_patterns.py` asserts the super-agent count exceeds 18; `tests/test_tools.py` asserts the focused sets are exactly 4.

**Distractors and why they fail:**

- *"Improve the tool descriptions."* Canonical trap. Better descriptions on 18 overlapping tools still cause attention fragmentation; the agent spends context budget evaluating descriptions rather than executing.
- *"Add a tool-selection preprocessing step."* You have added a second agent to hide the first agent's problem. Now you have two agents to debug and the underlying attention-fragmentation still exists during selection.
- *"Raise the temperature so the agent is less deterministic."* Actively harmful -- tool selection should be *more* deterministic, not less.

### Question 3 -- Silent Failure

**Scenario.** A research report is missing a critical source after an upstream API timeout. No error was flagged in the pipeline. What do you change?

**Correct answer.** Require **structured error context** from subagents. Every tool handler returns a `ToolErrorResponse` with `error_type`, `retry_eligible`, `fallback_available`, and `source`. This gives the coordinator a decision tree: retry transient failures, fallback for permanent errors, flag gaps in the final report otherwise.

**Where to see it.** Notebook 04 (`04_error_handling.ipynb`) -- `handle_fetch_page_silent` vs the structured `dispatch` on the same timeout URL, plus the 404 URL which produces a *different* decision tree. Verified in `tests/test_error_handling.py`.

**Distractors and why they fail:**

- *"Increase timeout duration."* Symptom fix. The next slow service still fails silently; you have just moved the threshold.
- *"Add retry logic."* Partially correct -- retry is *one branch* of the decision tree. It does not help when the failure is a 404 or a permanent auth error. You cannot build the full decision tree on a response that refuses to admit failure.
- *"Let the coordinator infer failures from response shape."* Requires the coordinator to know every subagent's internal contract. Brittle. The `ToolErrorResponse` schema makes the contract explicit and uniform.

## CCA Exam Tip

> The Multi-Agent Research System scenario draws from the three heaviest domains:
> - Agentic Architecture (27%)
> - Tool Design & MCP (18%)
> - Context Management & Reliability (15%)
>
> Together: **60% of the exam weight**. Master these patterns and you have a framework for every scenario.
>
> Key models to know:
> - `SubTask` -- the unit of delegation with explicit context
> - `ToolErrorResponse` -- structured errors for informed decision-making
> - `ConflictRecord` -- deterministic resolution metadata
> - `ResearchReport` -- transparent output with gaps and confidence